# jevmark: LoRA SFT for one backbone size (task 1.7)

One session trains one size. It clones the private repo at one commit, installs pinned dependencies, builds the data, runs a smoke training, the full training, then evaluates the trained adapter and re-evaluates the frozen base of the same size (decision 38). All logic lives in the repo; see `docs/KAGGLE.md`.

Settings: accelerator GPU T4 x2, internet on, secret `GITHUB_TOKEN` attached.

Fast cycle (decision 42): set `FAST = True` to train 0.6B for 300 steps and evaluate 300 records per split with every diagnostic, including `--shuffle-questions` on one split, into `runs/fast_<size>/` (gitignored), in about 25 minutes plus install. Run it on every new data version before a full session; `docs/KAGGLE.md` section 8 has the sequence.


In [ ]:
# Parameters: set before running. COMMIT must be a full 40-character sha.
REPO = "OWNER/jevmark"
COMMIT = "0000000000000000000000000000000000000000"
SIZE = "06b"          # "06b" or "17b"; one size per session
SMOKE_STEPS = 20      # optimizer steps in the smoke training
MAX_HOURS = 6.0       # training wall clock cap; leaves time for the two evaluations in a 9 hour session
FAST = False          # True: the fast cycle only, about 25 minutes plus install (docs/KAGGLE.md section 8); False: the full session
FAST_STEPS = 300      # optimizer steps in the fast cycle
FAST_RECORDS = 300    # records per split evaluated in the fast cycle (a seeded stratified sample)
FAST_SHUFFLE_SPLIT = "test_indomain"  # split evaluated a second time with its questions reordered


In [ ]:
# Clone REPO at COMMIT. The token reaches git only through environment variables,
# is never put on a command line or in .git/config, and is redacted from any output.
import base64
import os
import re
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

assert re.fullmatch(r"[\w.-]+/[\w.-]+", REPO), "REPO must be owner/name"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "COMMIT must be a full 40-character sha"
assert SIZE in ("06b", "17b"), "SIZE must be 06b or 17b"
# Let the CUDA caching allocator grow segments instead of fragmenting (decision 45). PyTorch 2.9 and later
# read PYTORCH_ALLOC_CONF; earlier releases read only PYTORCH_CUDA_ALLOC_CONF, so both are set. Every later
# !python inherits them.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path("/tmp/jevmark")  # outside /kaggle/working, so the notebook output holds only runs/


def run(cmd, env=None, secrets=()):
    result = subprocess.run(cmd, cwd=WORK, env=env, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for secret in secrets:
        output = output.replace(secret, "***")
    if output.strip():
        print(output[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"{cmd[0]} {cmd[1]} failed with exit code {result.returncode}")


token = UserSecretsClient().get_secret("GITHUB_TOKEN")
header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": header,
}
WORK.mkdir(parents=True, exist_ok=True)
if not (WORK / ".git").exists():
    run(["git", "init", "-q"])
    run(["git", "remote", "add", "origin", f"https://github.com/{REPO}.git"])
run(["git", "fetch", "-q", "--depth", "1", "origin", COMMIT], env=git_env, secrets=(token, header))
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"])
del token, header, git_env

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=WORK, capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, f"checked out {head}, expected {COMMIT}"
os.chdir(WORK)
print("checked out", head)

# Committed run directories are history, not state (decision 45): a fresh clone holds the summary and log of an
# earlier run, which must never pass for this session's result. Delete this size's training runs before training.
import shutil

for stale in (f"sft_{SIZE}", f"sft_{SIZE}_smoke", f"fast_{SIZE}"):
    shutil.rmtree(WORK / "runs" / stale, ignore_errors=True)
print("removed committed training run directories:", f"sft_{SIZE}, sft_{SIZE}_smoke, fast_{SIZE}")


In [ ]:
# The Hugging Face packages pinned to uv.lock; Kaggle keeps its own torch and numpy (decision 23). jevmark itself without deps.
# Kaggle's preinstalled torchao 0.10 makes peft 0.21 raise on LoRA adapter injection; jevmark does not use it.
!pip uninstall -y -q torchao
!pip install -q -r requirements-kaggle.txt
!pip install -q -e . --no-deps
!python -c "import sys, torch, transformers, peft, datasets; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'transformers', transformers.__version__, 'peft', peft.__version__, 'datasets', datasets.__version__)"

In [ ]:
# Fail-fast (decision 45): a failing !python does not stop a cell, so every training cell checks its own result
# with jevmark.runcheck and raises on failure, and every evaluation cell refuses to run unless its training passed.
import datetime
import json
import shutil

from jevmark.runcheck import require_training

TRAINED = {}  # "fast", "smoke", "full" -> True once that training cell printed TRAINING PASS


def evaluation_allowed(stage):
    if not TRAINED.get(stage):
        raise RuntimeError(f"refusing to evaluate: the {stage} training cell did not print TRAINING PASS")


def check_exit(what, code):
    if code != 0:
        raise RuntimeError(f"{what} exited with code {code}")


In [ ]:
# Build the nine splits; the build fails loudly if any build check fails. The leak probes and the duplicate
# check (make data) run locally before a commit is used here, so the notebook only builds (make data-build).
!make data-build PY=python


In [ ]:
# Fast cycle: FAST_STEPS steps on SIZE, then FAST_RECORDS records per split with every diagnostic, into runs/fast_{SIZE}/.
if FAST:
    FAST_RUN = f"fast_{SIZE}"
    started = datetime.datetime.now(datetime.timezone.utc)
    !python scripts/train_sft.py --config configs/sft_{SIZE}.yaml run_name={FAST_RUN} --limit-steps {FAST_STEPS} --device cuda
    training_exit = _exit_code
    shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)  # keep the state even if the check fails
    require_training(WORK / "runs" / FAST_RUN, started, training_exit)
    TRAINED["fast"] = True
    evaluation_allowed("fast")
    !python scripts/evaluate.py --ckpt runs/{FAST_RUN} --run-name {FAST_RUN} --limit {FAST_RECORDS} --shuffle-questions {FAST_SHUFFLE_SPLIT} --device cuda
    check_exit("evaluate.py", _exit_code)
    shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)
    metrics = json.loads((WORK / "runs" / FAST_RUN / "metrics.json").read_text())
    assert metrics["git"]["commit"] == COMMIT
    for split, m in metrics["splits"].items():
        o = m["overall"]
        print(f"{split:20} acc {o['accuracy']:.3f} [{o['accuracy_ci'][0]:.3f}, {o['accuracy_ci'][1]:.3f}]  ece {o['ece']:.3f}")
    print("order sensitivity", FAST_SHUFFLE_SPLIT, metrics["splits"][FAST_SHUFFLE_SPLIT]["order_sensitivity"]["overall"])
    print("fast cycle done; the remaining cells are skipped while FAST is True")


In [ ]:
# Smoke training: SMOKE_STEPS steps into runs/sft_{SIZE}_smoke/ (gitignored), including the pre-flight memory check,
# a validation and the final valid pass. Prints TRAINING PASS or TRAINING FAIL and raises on FAIL.
if not FAST:
    started = datetime.datetime.now(datetime.timezone.utc)
    !python scripts/train_sft.py --config configs/sft_{SIZE}.yaml run_name=sft_{SIZE}_smoke --limit-steps {SMOKE_STEPS} --device cuda
    require_training(WORK / "runs" / f"sft_{SIZE}_smoke", started, _exit_code)
    TRAINED["smoke"] = True


In [ ]:
# Full training into runs/sft_{SIZE}/. Prints TRAINING PASS or TRAINING FAIL and raises on FAIL, so no evaluation runs.
# If it stops at MAX_HOURS, runs/sft_{SIZE}/last holds the state for --resume (docs/KAGGLE.md section 7).
if not FAST:
    if not TRAINED.get("smoke"):
        raise RuntimeError("refusing to train: the smoke training cell did not print TRAINING PASS")
    started = datetime.datetime.now(datetime.timezone.utc)
    !python scripts/train_sft.py --config configs/sft_{SIZE}.yaml --max-hours {MAX_HOURS} --device cuda
    training_exit = _exit_code
    shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)  # keep the state even if the check fails
    require_training(WORK / "runs" / f"sft_{SIZE}", started, training_exit)
    TRAINED["full"] = True


In [ ]:
# Evaluate the trained adapter on all nine splits, with test_indomain also evaluated with its questions reordered:
# runs/sft_{SIZE}/metrics.json, results.jsonl.gz, plots/. Refuses to run unless full training printed TRAINING PASS.
if not FAST:
    evaluation_allowed("full")
    !python scripts/evaluate.py --ckpt runs/sft_{SIZE} --shuffle-questions test_indomain --device cuda
    check_exit("evaluate.py", _exit_code)


In [ ]:
# Re-evaluate the frozen base of the same size with this code version (decision 38): runs/base_{SIZE}/.
# Refuses to run unless full training printed TRAINING PASS, so a failed session stops here too.
if not FAST:
    evaluation_allowed("full")
    !python scripts/evaluate.py --ckpt base --config configs/base_{SIZE}.yaml --device cuda
    check_exit("evaluate.py", _exit_code)


In [ ]:
# Copy runs/ (including adapter/ and last/) to /kaggle/working/runs for download.
if not FAST:
    shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)
    for name in (f"sft_{SIZE}", f"base_{SIZE}"):
        metrics = json.loads((Path("/kaggle/working/runs") / name / "metrics.json").read_text())
        assert metrics["git"]["commit"] == COMMIT, name
        test = metrics["splits"]["test_indomain"]["overall"]
        print(name, "commit", metrics["git"]["commit"][:12], "dirty", metrics["git"]["dirty"], "fp32 fallback", metrics["precision"]["fp32_fallback_used"],
              f"test_indomain acc {test['accuracy']:.4f} ece {test['ece']:.4f}", f"{metrics['wall_clock_seconds'] / 60:.1f} min")
    print(json.loads((Path("/kaggle/working/runs") / f"sft_{SIZE}" / "train_summary.json").read_text()))
